# Enrichment Data: Download, Filter, and Feature Engineering
## Air Quality Prediction — Harris County, Houston TX

Covers four sources:
1. **EPA FRS** — facility locations with lat/lon
2. **EPA TRI** — annual toxic air emissions by facility
3. **EPA EJScreen** — block-group pollution burden and demographics
4. **TxDOT AADT** — annual average daily traffic counts

Each section: download → filter Harris County → build model features → save.

Final output: `data/enrichment/features_harris.csv` — one row per date,
ready to merge onto your main `houston_aqi_weather_combined.csv`.

In [ ]:
import pandas as pd
import numpy as np
import os, glob, requests, zipfile, time
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data/enrichment', exist_ok=True)

# Harris County identifiers
HARRIS_FIPS      = '48201'   # state (48) + county (201)
HARRIS_STATE     = '48'
HARRIS_COUNTY    = '201'
HOUSTON_LAT      = 29.7604
HOUSTON_LON      = -95.3698

# Date range matching your main dataset
DATE_MIN = '2022-01-01'
DATE_MAX = '2025-10-01'
YEARS    = [2022, 2023, 2024]   # TRI 2024 may not be released yet; drop if 404

---
## 1. EPA FRS — Facility Registry Service

**Source:** https://www.epa.gov/frs/epa-frs-facilities-state-single-file-csv-download  
**Direct download:** `https://ordsext.epa.gov/FLA/www3/state_files/texas.zip`

Contains every EPA-regulated facility in Texas with lat/lon, NAICS codes,
and which EPA programs it's enrolled in (TRI, RMP, NPDES, etc.).

### What we extract
- All Harris County facilities with valid coordinates
- Flag: is it a TRI reporter? RMP (risk management plan = high-hazard)?
- Industry type via NAICS prefix

In [ ]:
FRS_URL   = 'https://ordsext.epa.gov/FLA/www3/state_files/texas.zip'
FRS_LOCAL = 'data/enrichment/frs_texas.zip'

if not os.path.exists(FRS_LOCAL):
    print('Downloading FRS Texas (~30 MB)...')
    r = requests.get(FRS_URL, stream=True, timeout=120)
    r.raise_for_status()
    with open(FRS_LOCAL, 'wb') as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
    print('Done.')
else:
    print('FRS zip already downloaded.')

with zipfile.ZipFile(FRS_LOCAL) as z:
    csv_name = [n for n in z.namelist() if n.lower().endswith('.csv')][0]
    with z.open(csv_name) as f:
        frs_tx = pd.read_csv(f, dtype=str, low_memory=False)

print(f'Texas FRS rows: {len(frs_tx):,}')
print('Columns:', frs_tx.columns.tolist())

In [ ]:
# Filter Harris County and clean coordinates
frs = frs_tx[frs_tx['COUNTY_NAME'].str.upper().str.strip() == 'HARRIS'].copy()
frs['lat'] = pd.to_numeric(frs['LATITUDE83'],  errors='coerce')
frs['lon'] = pd.to_numeric(frs['LONGITUDE83'], errors='coerce')
frs = frs.dropna(subset=['lat', 'lon'])

# Flag program enrollment from PGM_SYS_ACRNMS column (pipe/comma separated)
frs['is_tri'] = frs['PGM_SYS_ACRNMS'].str.contains('TRIS',   na=False).astype(int)
frs['is_rmp'] = frs['PGM_SYS_ACRNMS'].str.contains('RMP',    na=False).astype(int)
frs['is_air'] = frs['PGM_SYS_ACRNMS'].str.contains('ICIS-AIR', na=False).astype(int)

# NAICS 2-digit prefix for industry sector
frs['naics2'] = frs['NAICS_CODES'].str[:2]

keep = ['REGISTRY_ID', 'PRIMARY_NAME', 'lat', 'lon',
        'naics2', 'is_tri', 'is_rmp', 'is_air', 'PGM_SYS_ACRNMS']
frs = frs[[c for c in keep if c in frs.columns]]

print(f'Harris County facilities with coordinates: {len(frs):,}')
print(f'  TRI reporters: {frs["is_tri"].sum()}')
print(f'  RMP facilities: {frs["is_rmp"].sum()}')
print(f'  Air permit holders: {frs["is_air"].sum()}')
frs.head()

In [ ]:
# ── FRS Feature Engineering ──────────────────────────────────────────────────
#
# FRS is static (no year column), so we produce scalar features that get
# broadcast across every row of your time-series dataset.
#
# Spatial approach: for each of the ~20 AQS monitoring stations in Harris County,
# compute how many facilities sit within a radius. Here we use a fast
# haversine approximation — no GeoPandas required.

def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorised haversine distance in km."""
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def count_facilities_within(facilities_df, center_lat, center_lon, radius_km):
    """Count facilities and sum boolean flags within radius_km of a point."""
    dist = haversine_km(center_lat, center_lon,
                        facilities_df['lat'].values,
                        facilities_df['lon'].values)
    nearby = facilities_df[dist <= radius_km]
    return {
        f'frs_total_{radius_km}km':   len(nearby),
        f'frs_tri_{radius_km}km':     int(nearby['is_tri'].sum()) if 'is_tri' in nearby else 0,
        f'frs_rmp_{radius_km}km':     int(nearby['is_rmp'].sum()) if 'is_rmp' in nearby else 0,
        f'frs_air_{radius_km}km':     int(nearby['is_air'].sum()) if 'is_air' in nearby else 0,
    }

# County centroid as a simple single-point proxy.
# If you have your actual AQS station coordinates, loop over those instead
# and average the results — or keep per-station features.
frs_feats_5km  = count_facilities_within(frs, HOUSTON_LAT, HOUSTON_LON, 5)
frs_feats_10km = count_facilities_within(frs, HOUSTON_LAT, HOUSTON_LON, 10)
frs_feats_25km = count_facilities_within(frs, HOUSTON_LAT, HOUSTON_LON, 25)

frs_features = {**frs_feats_5km, **frs_feats_10km, **frs_feats_25km}
print('FRS proximity features (county centroid):')
for k, v in frs_features.items():
    print(f'  {k}: {v}')

frs.to_csv('data/enrichment/frs_harris.csv', index=False)

---
## 2. EPA TRI — Toxic Release Inventory

**Source:** https://www.epa.gov/toxics-release-inventory-tri-program/tri-basic-data-files-calendar-years-1987-present  
**Download pattern:** `https://www.epa.gov/system/files/other-files/2024-07/tri{YY}_TX.zip`

Tab-delimited file of facility × chemical rows. Each row is one chemical
reported by one facility for one year. We aggregate to total air emissions
per facility per year, then to county-level annual totals.

### What we extract
- `5.1_FUGITIVE_AIR` + `5.2_STACK_AIR` → total air releases in lbs
- Aggregated to Harris County annual total and facility count
- Joined to main dataset on year

In [ ]:
def download_tri(year):
    yy = str(year)[2:]
    # URL format changed in 2023 — try both patterns
    urls = [
        f'https://www.epa.gov/system/files/other-files/2024-07/tri{yy}_TX.zip',
        f'https://www.epa.gov/system/files/other-files/2023-07/tri{yy}_TX.zip',
    ]
    local = f'data/enrichment/tri{year}_TX.zip'
    if os.path.exists(local):
        print(f'  TRI {year}: already downloaded')
        return local
    for url in urls:
        r = requests.get(url, timeout=60)
        if r.status_code == 200:
            with open(local, 'wb') as f:
                f.write(r.content)
            print(f'  TRI {year}: downloaded from {url}')
            return local
        time.sleep(0.5)
    print(f'  TRI {year}: not available yet (HTTP {r.status_code}). '
          f'Download manually from the TRI page and save as {local}')
    return None

def load_tri(local_zip):
    with zipfile.ZipFile(local_zip) as z:
        fname = [n for n in z.namelist() if n.endswith('.txt') or n.endswith('.csv')][0]
        with z.open(fname) as f:
            df = pd.read_csv(f, sep='\t', dtype=str, low_memory=False)
    df.columns = [c.strip().upper().replace(' ', '_') for c in df.columns]
    return df

print('Downloading TRI data...')
tri_frames = []
for yr in YEARS:
    path = download_tri(yr)
    if path and os.path.exists(path):
        df = load_tri(path)
        # County column name varies by year: COUNTY, BIA_COUNTY, COUNTY_NAME
        county_col = next((c for c in ['COUNTY', 'BIA_COUNTY', 'COUNTY_NAME'] if c in df.columns), None)
        if county_col:
            df = df[df[county_col].str.upper().str.strip() == 'HARRIS']
        tri_frames.append(df)
        print(f'    Harris rows for {yr}: {len(df):,}')

In [ ]:
if not tri_frames:
    print('No TRI data loaded — check downloads above.')
else:
    tri = pd.concat(tri_frames, ignore_index=True)

    # Numeric air emission columns
    # Column names: '5.1_FUGITIVE_AIR' and '5.2_STACK_AIR' in basic file format
    fug_col   = next((c for c in tri.columns if 'FUGITIVE' in c), None)
    stack_col = next((c for c in tri.columns if 'STACK' in c and 'AIR' in c), None)
    year_col  = next((c for c in tri.columns if c in ['YEAR', 'REPORTING_YEAR']), None)
    lat_col   = next((c for c in tri.columns if 'LATITUDE' in c), None)
    lon_col   = next((c for c in tri.columns if 'LONGITUDE' in c), None)

    print(f'Air columns found: fugitive={fug_col}, stack={stack_col}')

    for c in [fug_col, stack_col, lat_col, lon_col]:
        if c:
            tri[c] = pd.to_numeric(tri[c], errors='coerce')

    # Total air releases per row (lbs)
    tri['air_lbs'] = tri[fug_col].fillna(0) + tri[stack_col].fillna(0) if (fug_col and stack_col) else np.nan

    tri.to_csv('data/enrichment/tri_harris.csv', index=False)
    print(f'Saved tri_harris.csv: {len(tri):,} facility-chemical rows')
    print(tri[[year_col, 'FACILITY_NAME', lat_col, lon_col, 'air_lbs']].head())

In [ ]:
# ── TRI Feature Engineering ───────────────────────────────────────────────────
#
# TRI is annual, so TRI features vary by year but not by day.
# We produce one row per year, then merge onto the main dataset on year.

if tri_frames:
    tri_annual = (
        tri.groupby(year_col)
        .agg(
            tri_total_air_lbs       = ('air_lbs', 'sum'),
            tri_facility_count      = ('FACILITY_NAME', 'nunique'),
            tri_max_facility_lbs    = ('air_lbs', 'max'),   # single largest emitter
        )
        .reset_index()
        .rename(columns={year_col: 'year'})
    )
    tri_annual['year'] = tri_annual['year'].astype(int)

    # Log-transform total emissions (heavy right skew)
    tri_annual['tri_log_air_lbs'] = np.log1p(tri_annual['tri_total_air_lbs'])

    print('TRI annual features for Harris County:')
    print(tri_annual)

    # ── Spatial: emissions within radius of county centroid ──────────────────
    # Aggregate per-facility first, then apply radius filter per year
    facility_year = (
        tri[tri['air_lbs'].notna() & tri[lat_col].notna()]
        .groupby([year_col, 'FACILITY_NAME', lat_col, lon_col])
        ['air_lbs'].sum()
        .reset_index()
    )
    facility_year.columns = ['year', 'facility', 'lat', 'lon', 'air_lbs']

    rows = []
    for yr, grp in facility_year.groupby('year'):
        dist = haversine_km(HOUSTON_LAT, HOUSTON_LON, grp['lat'].values, grp['lon'].values)
        rows.append({
            'year': int(yr),
            'tri_air_lbs_25km': grp.loc[dist <= 25, 'air_lbs'].sum(),
            'tri_n_facilities_25km': int((dist <= 25).sum()),
        })
    tri_spatial = pd.DataFrame(rows)
    tri_annual = tri_annual.merge(tri_spatial, on='year', how='left')

    print('\nWith spatial features:')
    print(tri_annual)

---
## 3. EPA EJScreen — Environmental Justice Screening

**Source (FTP):** https://gaftp.epa.gov/EJSCREEN/2024/2.32_September_UseMe/  
**Direct file:** `EJSCREEN_2024_BG_with_AS_CNMI_GU_VI.csv.zip` (~250 MB)

Census block-group level. Filter on `ID` starting with `48201`.

### What we extract
- `PM25`, `OZONE` — modeled ambient concentrations
- `PTRAF` — traffic proximity index
- `CANCER`, `RESP` — air toxics risk scores
- `MINORPCT`, `LOWINCPCT` — demographic vulnerability
- `P_EJ_DSIEO` — composite EJ index percentile

EJScreen is static (one snapshot per year), so like FRS it produces
scalar features broadcast across all dates.

In [ ]:
EJSCREEN_URL = (
    'https://gaftp.epa.gov/EJSCREEN/2024/2.32_September_UseMe/'
    'EJSCREEN_2024_BG_with_AS_CNMI_GU_VI.csv.zip'
)
EJSCREEN_LOCAL = 'data/enrichment/ejscreen_2024_bg.csv.zip'

if not os.path.exists(EJSCREEN_LOCAL):
    print('Downloading EJScreen 2024 block groups (~250 MB)...')
    r = requests.get(EJSCREEN_URL, stream=True, timeout=600)
    r.raise_for_status()
    with open(EJSCREEN_LOCAL, 'wb') as f:
        for chunk in r.iter_content(65536):
            f.write(chunk)
    print('Done.')
else:
    print('EJScreen already downloaded.')

with zipfile.ZipFile(EJSCREEN_LOCAL) as z:
    csv_name = [n for n in z.namelist() if n.lower().endswith('.csv')][0]
    with z.open(csv_name) as f:
        ej = pd.read_csv(f, dtype={'ID': str}, low_memory=False)

print(f'National EJScreen block groups: {len(ej):,}')

In [ ]:
# Filter Harris County: block group IDs are 12-digit strings starting with 48201
ej_harris = ej[ej['ID'].str.startswith('48201')].copy()

EJ_KEEP = [
    'ID',
    'PM25', 'OZONE',
    'PTRAF', 'PNPL', 'PRMP', 'PTSDF',
    'CANCER', 'RESP',
    'MINORPCT', 'LOWINCPCT', 'LINGISOPCT',
    'DEMOGIDX_2', 'P_EJ_DSIEO',
]
available = [c for c in EJ_KEEP if c in ej_harris.columns]
ej_harris = ej_harris[available]

# Convert to numeric
for c in available[1:]:
    ej_harris[c] = pd.to_numeric(ej_harris[c], errors='coerce')

print(f'Harris County block groups: {len(ej_harris)}')
print(ej_harris.describe().round(3))

ej_harris.to_csv('data/enrichment/ejscreen_harris_2024.csv', index=False)

In [ ]:
# ── EJScreen Feature Engineering ─────────────────────────────────────────────
#
# Collapse all Harris block groups to county-level aggregates.
# Mean captures typical burden; 90th pctl captures where the worst hotspots are.
#
# These are static scalars — broadcast to every row of your time-series.

ej_features = {}
for col in ['PM25', 'OZONE', 'PTRAF', 'CANCER', 'RESP', 'P_EJ_DSIEO']:
    if col in ej_harris.columns:
        ej_features[f'ej_{col.lower()}_mean']  = ej_harris[col].mean()
        ej_features[f'ej_{col.lower()}_p90']   = ej_harris[col].quantile(0.90)

# Demographic burden summary
for col in ['MINORPCT', 'LOWINCPCT']:
    if col in ej_harris.columns:
        ej_features[f'ej_{col.lower()}_mean'] = ej_harris[col].mean()

# Count of block groups in highest EJ burden (top 20% nationally)
if 'P_EJ_DSIEO' in ej_harris.columns:
    ej_features['ej_high_burden_bg_count'] = int((ej_harris['P_EJ_DSIEO'] >= 80).sum())

print('EJScreen county-level features:')
for k, v in ej_features.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

---
## 4. TxDOT AADT — Annual Average Daily Traffic

**Source:** https://gis-txdot.opendata.arcgis.com/datasets/txdot-annual-average-daily-traffic-counts-public  

Two ways to get the data:
- **Manual**: go to the portal, click Download → Spreadsheet (CSV). Save as `data/enrichment/txdot_aadt_statewide.csv`.
- **API** (done below): query the ArcGIS REST endpoint directly for Harris County only.

Harris County = `COUNTY_NBR == 101` in TxDOT's system.

### What we extract
- AADT per road segment point with lat/lon
- Historical AADT columns (one per year: `AADT_2022`, `AADT_2023`, etc.)
- Weighted average AADT within radius of county centroid

In [ ]:
TXDOT_ENDPOINT = (
    'https://services.arcgis.com/KTcxiTD9dsQw4r7Z/arcgis/rest/services/'
    'TxDOT_AADT/FeatureServer/0/query'
)
TXDOT_LOCAL = 'data/enrichment/txdot_aadt_harris.csv'

if not os.path.exists(TXDOT_LOCAL):
    print('Fetching TxDOT AADT for Harris County via ArcGIS REST API...')
    all_rows = []
    offset   = 0
    batch    = 1000

    while True:
        params = {
            'where':             'COUNTY_NBR=101',
            'outFields':         '*',
            'f':                 'json',
            'resultOffset':      offset,
            'resultRecordCount': batch,
            'returnGeometry':    'true',
        }
        r = requests.get(TXDOT_ENDPOINT, params=params, timeout=60)
        data     = r.json()
        features = data.get('features', [])
        if not features:
            break
        for feat in features:
            row = feat.get('attributes', {})
            geom = feat.get('geometry', {})
            row['LON'] = geom.get('x')
            row['LAT'] = geom.get('y')
            all_rows.append(row)
        print(f'  Fetched {len(all_rows)} records...')
        if len(features) < batch:
            break
        offset += batch
        time.sleep(0.3)

    if all_rows:
        aadt = pd.DataFrame(all_rows)
        aadt.to_csv(TXDOT_LOCAL, index=False)
        print(f'Saved {len(aadt):,} AADT points for Harris County.')
    else:
        print('No data returned. Try manual download from the TxDOT open data portal.')
        aadt = pd.DataFrame()
else:
    aadt = pd.read_csv(TXDOT_LOCAL)
    print(f'Loaded TXDOT AADT from cache: {len(aadt):,} records')

if not aadt.empty:
    aadt_yr_cols = sorted([c for c in aadt.columns if c.startswith('AADT_') and c[5:].isdigit()])
    print('Available AADT year columns:', aadt_yr_cols)
    print(aadt[['RTE_NM', 'LAT', 'LON'] + aadt_yr_cols[:3]].head())

In [ ]:
# ── AADT Feature Engineering ──────────────────────────────────────────────────
#
# AADT is annual (one value per segment per year), so features vary by year.
# We compute county-level traffic volume summary per year.

if not aadt.empty and aadt_yr_cols:
    aadt['LAT'] = pd.to_numeric(aadt['LAT'], errors='coerce')
    aadt['LON'] = pd.to_numeric(aadt['LON'], errors='coerce')
    aadt_clean = aadt.dropna(subset=['LAT', 'LON'])

    # Distance from county centroid
    dist = haversine_km(HOUSTON_LAT, HOUSTON_LON,
                        aadt_clean['LAT'].values,
                        aadt_clean['LON'].values)

    aadt_rows = []
    for yr_col in aadt_yr_cols:
        yr = int(yr_col.split('_')[1])
        if yr not in YEARS:
            continue
        vals = pd.to_numeric(aadt_clean[yr_col], errors='coerce')
        nearby_mask = dist <= 25
        aadt_rows.append({
            'year':                    yr,
            'aadt_mean_25km':          vals[nearby_mask].mean(),
            'aadt_median_25km':        vals[nearby_mask].median(),
            'aadt_p90_25km':           vals[nearby_mask].quantile(0.90),
            'aadt_total_veh_25km':     vals[nearby_mask].sum(),      # proxy for total VMT burden
            'aadt_n_segments_25km':    int(nearby_mask.sum()),
        })

    aadt_annual = pd.DataFrame(aadt_rows)
    # Log-transform total vehicles (heavy skew)
    aadt_annual['aadt_log_total_25km'] = np.log1p(aadt_annual['aadt_total_veh_25km'])

    print('AADT annual features for Harris County (within 25 km):')
    print(aadt_annual.round(1))

---
## 5. Assemble Final Enrichment Feature Table

Merge all enrichment features into a single table with one row per date,
ready to join onto your main `houston_aqi_weather_combined.csv`.

- **Static features** (FRS, EJScreen) → same value for every date
- **Annual features** (TRI, AADT) → value changes by year, constant within a year

Strategy: build a date-spine from your main dataset, attach annual features
by year, broadcast static features as constants.

In [ ]:
# Build date spine matching main dataset
dates = pd.date_range(DATE_MIN, DATE_MAX, freq='D')
feat_df = pd.DataFrame({'Date': dates})
feat_df['year'] = feat_df['Date'].dt.year

# ── Attach annual features (TRI + AADT) ──────────────────────────────────────
if 'tri_annual' in dir() and tri_annual is not None:
    feat_df = feat_df.merge(tri_annual, on='year', how='left')
    print('Merged TRI annual features.')

if 'aadt_annual' in dir() and not aadt_annual.empty:
    feat_df = feat_df.merge(aadt_annual, on='year', how='left')
    print('Merged AADT annual features.')

# ── Broadcast static features (FRS + EJScreen) ───────────────────────────────
# These don't change over time, so just assign as columns
for k, v in frs_features.items():
    feat_df[k] = v

for k, v in ej_features.items():
    feat_df[k] = v

# Drop the year helper column (already in main dataset)
feat_df = feat_df.drop(columns=['year'])

print(f'\nEnrichment feature table shape: {feat_df.shape}')
print('Columns:', feat_df.columns.tolist())
feat_df.head()

In [ ]:
# Save enrichment features
feat_df.to_csv('data/enrichment/features_harris.csv', index=False)
print('Saved: data/enrichment/features_harris.csv')

# ── Merge onto main dataset ───────────────────────────────────────────────────
main_path = 'data/processed/houston_aqi_weather_combined.csv'
if os.path.exists(main_path):
    main = pd.read_csv(main_path, parse_dates=['Date'])
    combined = main.merge(feat_df, on='Date', how='left')
    combined.to_csv('data/processed/houston_aqi_full.csv', index=False)
    print(f'Merged dataset saved: data/processed/houston_aqi_full.csv')
    print(f'Shape: {combined.shape}')
    print(f'New columns added: {combined.shape[1] - main.shape[1]}')
else:
    print(f'{main_path} not found — run data_collection.ipynb first.')
    print('Enrichment features are saved separately in data/enrichment/features_harris.csv')

---
## Feature Reference

| Feature | Source | Type | Notes |
|---|---|---|---|
| `frs_total_5km` | FRS | static int | All EPA facilities within 5 km of centroid |
| `frs_tri_5km` | FRS | static int | TRI-reporting facilities within 5 km |
| `frs_rmp_5km` | FRS | static int | High-hazard (RMP) facilities within 5 km |
| `frs_total_10km` | FRS | static int | Same, 10 km radius |
| `frs_total_25km` | FRS | static int | Same, 25 km radius |
| `tri_total_air_lbs` | TRI | annual float | Total county air emissions (lbs) |
| `tri_log_air_lbs` | TRI | annual float | Log1p of above — use this in models |
| `tri_facility_count` | TRI | annual int | Unique TRI-reporting facilities |
| `tri_air_lbs_25km` | TRI | annual float | Air emissions within 25 km |
| `ej_pm25_mean` | EJScreen | static float | Mean modeled PM2.5 across block groups |
| `ej_pm25_p90` | EJScreen | static float | 90th pctl PM2.5 (hotspot indicator) |
| `ej_cancer_mean` | EJScreen | static float | Mean air toxics cancer risk |
| `ej_ptraf_mean` | EJScreen | static float | Mean traffic proximity index |
| `ej_p_ej_dsieo_mean` | EJScreen | static float | Mean EJ index percentile |
| `ej_high_burden_bg_count` | EJScreen | static int | Block groups in top 20% nationally |
| `aadt_mean_25km` | TxDOT | annual float | Mean AADT on road segments within 25 km |
| `aadt_p90_25km` | TxDOT | annual float | 90th pctl AADT (busiest roads) |
| `aadt_log_total_25km` | TxDOT | annual float | Log total vehicle volume within 25 km |

### Tips for using these in your model

**Static features have zero variance** — tree-based models (Random Forest, XGBoost)
will ignore them automatically, but they can still help linear models capture
background baseline levels. Consider dropping them from tree models to reduce noise.

**Annual features are low-variance** — TRI and AADT only change year-to-year.
They add more value as context than as predictive signal. The log-transformed
versions reduce the influence of extreme emission years.

**If you have AQS station coordinates**, replace `HOUSTON_LAT/LON` with per-station
coordinates and run the radius calculations per station. This gives you
station-specific exposure features that will be much more informative.